In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
from pathlib import Path
import openslide
import PIL.Image
import tifffile
import re

import sys
sys.path.append("/workspaces/TRIDENT")


from trident.wsi_objects.tiles import Tile, AggregatedTile
from trident.wsi_objects.utils import (
    extract_tile_size,
    extract_page_infos,
    extract_JPEGTables,
    extract_root_image_size,
)


In [ ]:
import torch
from torchvision.io import decode_jpeg, ImageReadMode
import torchvision.transforms.v2 as transforms

def extract_aggregated_tile(slide_path: Path, subtile_offsets: list[int], subtile_bytecounts: list[int], coord: np.ndarray, tile_size: int, subtile_size: int) -> dict:
    subtiles: list[torch.Tensor] = [None] * len(subtile_offsets)

    fd = os.open(slide_path, os.O_RDONLY | os.O_DIRECT)

    # Process aggregated tiles
    for i, (subtile_offset, subtile_bytecount) in enumerate(
        zip(subtile_offsets, subtile_bytecounts)
    ):
        tile_bytes = os.pread(fd, subtile_bytecount, subtile_offset)
        subtiles[i] = torch.frombuffer(tile_bytes, dtype=torch.uint8)

    os.close(fd)

    output = dict(
        tile=subtiles,
        coord=coord,
        aggregated=True,
        subtile_size=subtile_size,
        tile_size=tile_size,
        slide_path=slide_path,
    )
    return output


def process_tiles(tile_infos: dict):
        """
        Decompress and augment the tile in a separate CUDA stream.
        """
        tile_size = int(tile_infos["tile_size"])
        # -- Create destination tensor on GPU
        tile_tensor = torch.zeros(
            (3, tile_size, tile_size),
            dtype=torch.uint8,
        )
        # ========== aggregated sub-tiles ==========
        if tile_infos.get("aggregated", False):
            subtile_size = int(tile_infos["subtile_size"])
            compressed_subtiles = tile_infos["tile"]
            decoded_subtiles = decode_jpeg(
                compressed_subtiles, mode=ImageReadMode.UNCHANGED
            )
            grid_size = tile_size // subtile_size
            # Iterate over each compressed tile in the batch
            for subtile_idx in range(grid_size * grid_size):
                # Calculate the row and column position of the subtile in the grid
                row_idx = subtile_idx // grid_size
                col_idx = subtile_idx % grid_size
                # Calculate the pixel range for the current subtile in the full tile
                row_start = row_idx * subtile_size
                row_end = row_start + subtile_size
                col_start = col_idx * subtile_size
                col_end = col_start + subtile_size
                # Place the decoded subtile in the correct position in the full tile tensor
                tile_tensor[
                    :,
                    row_start:row_end,
                    col_start:col_end,
                ] = decoded_subtiles[subtile_idx]
        else:
            # ========== native tiles ==========
            decode_jpeg(
                tile_infos["tile"], mode=ImageReadMode.UNCHANGED
            )
        tile_tensor = transforms.functional.to_image(tile_tensor)

        return tile_tensor.unsqueeze(0)


def load_aggregated_tile(
    slide_path: Path,
    coord: np.ndarray,
    offsets: list[int],
    bytcounts: list[int],
    tile_size: int,
    subtile_size: int,
) -> PIL.Image.Image:
    """
    Load an aggregated tile from a slide.
    """

    # Extract the aggregated tile
    tile_infos = extract_aggregated_tile(
        slide_path, offsets, bytcounts, coord, tile_size, subtile_size
    )
    tile = process_tiles(tile_infos)

    image = transforms.functional.to_pil_image(tile.squeeze(0))

    return image



def load_openslide_tile(
    slide_path: Path,
    coord: np.ndarray,
    level: int,
    tile_size: int,
) -> PIL.Image.Image:
    """
    Load a tile from a slide using OpenSlide.
    """

    # Extract the tile
    slide = openslide.OpenSlide(slide_path)
    image = slide.read_region(
        (coord[0], coord[1]), level, (tile_size, tile_size)
    )
    return image


In [ ]:
import h5py
import math

def show_hdf5(hdf5_path: Path):
    """
    Show the contents of an HDF5 file.
    """
    with h5py.File(hdf5_path, "r") as f:
        for key in f.keys():
            print(f"{key}: {f[key].shape}")
            if isinstance(f[key], h5py.Dataset):
                print(f"{key} data: {f[key][:]}")
            print(f"{key} attributes: {f[key]}")
            for attr_key in f[key].attrs.keys():
                print(f"  {attr_key}: {f[key].attrs[attr_key]}")
        

def get_debugging_input(slide_path: Path, patches_path: Path, level: int):
    with h5py.File(patches_path, "r") as f:
        coords = f["coords"][:]
        patch_size = f["coords"].attrs["patch_size"]
        patch_size_level0 = f["coords"].attrs["patch_size_level0"]
        level0_magnification = f["coords"].attrs["level0_magnification"]
        target_magnification = f["coords"].attrs["target_magnification"]
        overlap = f["coords"].attrs["overlap"]
        name = f["coords"].attrs["name"]
        savetodir = f["coords"].attrs["savetodir"]

    assets = {"coords": np.array(coords)}
    attributes = {
        "patch_size": patch_size,  # Reference frame: patch_level
        "patch_size_level0": patch_size_level0,  # Reference frame: level0
        "level0_magnification": level0_magnification,
        "target_magnification": target_magnification,
        "overlap": overlap,
        "name": name,
        "savetodir": savetodir,
    }
    return slide_path, patches_path, assets, attributes, level


def coord_to_tileindex(
    coord: np.ndarray, tile_size: int, slide_width: int, slide_height: int
) -> int:
    """
    Convert a coordinate to a tile index.
    """
    tiles_x = math.ceil(slide_width / tile_size)
    x, y = coord
    x = int(x / tile_size)
    y = int(y / tile_size)
    return y * tiles_x + x


def coords_to_tiles(
    slide_path: Path,
    coords: np.ndarray,
    patch_size: int,
    level: int,
) -> list:
    """
    Convert pixel slide coordinates to tiles.
    """
    slide_infos = extract_page_infos(slide_path, level)
    slide_width, slide_height = slide_infos["width"], slide_infos["height"]
    tile_size = slide_infos["tile_size"]
    tiles_x = math.ceil(slide_width / tile_size)

    offsets = slide_infos["offsets"]
    bytecounts = slide_infos["bytecounts"]

    tiles = []
    if patch_size == tile_size:
        # Create normal tile for each coordinate
        for coord in coords:
            tile_index = coord_to_tileindex(coord, tile_size, slide_width, slide_height)
            tile_offset = offsets[tile_index]
            tile_bytecount = bytecounts[tile_index]
            tiles.append(Tile(tile_offset, tile_bytecount, coord, tile_index))
    elif patch_size % tile_size == 0:
        # Create an aggregate tile for each coordinate
        factor = patch_size // tile_size
        for coord in coords:
            tile_index = coord_to_tileindex(coord, tile_size, slide_width, slide_height)
            subtile_offsets = []
            subtile_bytecounts = []
            for i in range(factor):
                for j in range(factor):
                    subtile_index = tile_index + i * tiles_x + j
                    subtile_offsets.append(offsets[subtile_index])
                    subtile_bytecounts.append(bytecounts[subtile_index])
            tiles.append(
                AggregatedTile(subtile_offsets, subtile_bytecounts, coord, tile_index)
            )
    else:
        raise ValueError("invalid patch_size")
    return tiles


def save_entropy_patches(
    slide_path: Path,
    patches_path: Path,
    assets: dict,
    attributes: dict,
    level: int,
):
    """
    Convert patch coords to format readable by tile-aligned dataloader.

    Args:
        slide_path (Path): Path to the whole-slide image file.
        clam_patches_path (Path): Path to the HDF5 file containing CLAM patches.
        output_dir (Path): Directory to save the output HDF5 file.
        assets (dict): Dictionary containing the original coords.
        attributes (dict): Dictionary containing the original attributes.
        level (int): Extraction level used by the patcher.
    """

    # load CLAM coordinates

    coords = assets["coords"]
    patch_size = attributes["patch_size"]

    # Extract slide info
    slide_infos = extract_page_infos(slide_path, level)
    slide_width, slide_height = slide_infos["width"], slide_infos["height"]
    print(f"Slide size: {slide_width} x {slide_height}")
    original_tile_size = slide_infos["tile_size"]
    if patch_size < original_tile_size:
        raise ValueError(
            f"Patch size {patch_size} is smaller than original tile size {original_tile_size}."
        )
    if patch_size % original_tile_size != 0:
        # make the patch_size a multiple of the original tile size
        patch_size = math.ceil(patch_size / original_tile_size) * original_tile_size
        print(
            f"Patch size is not a multiple of the native tile size. Adjusted patch size to {patch_size}."
        )
        attributes["patch_size"] = patch_size
        if (
            "level0_magnification" in attributes
            and "target_magnification" in attributes
        ):
            attributes["patch_size_level0"] = (
                patch_size
                * attributes["level0_magnification"]
                // attributes["target_magnification"]
            )

    if level != 0:
        # convert root level coordinates to the specified level
        root_width, root_height = extract_root_image_size(slide_path)
        print(f"Root size: {root_width} x {root_height}")
        width_ratio = slide_width / root_width
        height_ratio = slide_height / root_height
        coords = np.floor(coords * np.array([width_ratio, height_ratio])).astype(
            np.int32
        )
        # we will have to revert the coordinates to the root level later
        revert_coords = lambda coords: np.floor(
            coords / np.array([width_ratio, height_ratio])
        ).astype(np.int32)
    elif np.any(coords[:, 0] >= slide_width) or np.any(coords[:, 1] >= slide_height):
        # check whether the coords are within the slide boundaries
        raise ValueError(
            "Coordinates exceed slide boundaries. Please check the slide level."
        )
    else:
        revert_coords = lambda coords: coords

    # Ensure coords are tile-aligned (multiple of original_tile_size)
    backward_aligned_coords = (coords // original_tile_size) * original_tile_size
    forward_aligned_coords = (
        np.ceil(coords / original_tile_size).astype(int) * original_tile_size
    )

    # Choose the closer alignment (backward or forward)
    dist_backward = np.abs(coords - backward_aligned_coords)
    dist_forward = np.abs(coords - forward_aligned_coords)

    # Use backward alignment if it's closer, otherwise use forward alignment
    aligned_coords = np.where(
        dist_backward <= dist_forward, backward_aligned_coords, forward_aligned_coords
    )

    # Filter out coordinates that exceed the valid slide boundaries
    valid_coords = aligned_coords[
        (aligned_coords[:, 0] <= slide_width - patch_size)
        & (aligned_coords[:, 1] <= slide_height - patch_size)
    ]

    print(
        f"Original coordinates: {len(coords)}\n",
        f"Aligned coordinates: {len(aligned_coords)}\n",
        f"Valid coordinates: {len(valid_coords)}\n",
    )

    print(valid_coords.shape)
    # valid_coords = np.array([16384, 16384])

    # Convert aligned coordinates to tiles
    tiles = coords_to_tiles(slide_path, valid_coords, patch_size, level)
    # pick 5 random indices to visualize
    random_indices = np.random.choice(
        len(tiles), size=min(5, len(tiles)), replace=False
    )
    
    tiles = [tiles[i] for i in random_indices]



    # plot all tiles to compare loading methods
    fig, axs = plt.subplots(len(tiles), 2, figsize=(10, 5 * len(tiles)))
    for i, tile in enumerate(tiles):
        direct_image = load_aggregated_tile(
            slide_path,
            tile.coord,
            tile.offset,
            tile.bytecount,
            patch_size,
            original_tile_size,
        )
        
        axs[i, 0].imshow(direct_image)
        axs[i, 0].set_title(f"Direct Load {i}, {tile.coord.tolist()}")
        axs[i, 0].axis("off")
        # get level0 coord 
        level0_coord = revert_coords(tile.coord)
        openslide_image = load_openslide_tile(
            slide_path,
            level0_coord,
            level,
            patch_size,
        )
        axs[i, 1].imshow(openslide_image)
        axs[i, 1].set_title(f"OpenSlide Load {i}, {tile.coord.tolist()}")
        axs[i, 1].axis("off")
    
    fig.tight_layout()
    fig.show()
    

    # # Extract JPEG tables
    # jpeg_tables = extract_JPEGTables(slide_path)

    # Save results to an HDF5 file
    # with h5py.File(patches_path, "w") as f:
    #     num_subtiles = int((patch_size // original_tile_size) ** 2)
    #     # Prepare datasets
    #     offsets = np.array([tile.offset for tile in tiles], dtype=np.uint64).reshape(
    #         -1, num_subtiles
    #     )
    #     f.create_dataset("offsets", data=offsets, dtype="uint64")

    #     bytecounts = np.array(
    #         [tile.bytecount for tile in tiles], dtype=np.uint32
    #     ).reshape(-1, num_subtiles)
    #     f.create_dataset("bytecounts", data=bytecounts, dtype="uint32")

    #     coords = revert_coords(
    #         np.array([tile.coord for tile in tiles], dtype=np.uint32).reshape(-1, 2)
    #     )
    #     coord_dataset = f.create_dataset("coords", data=coords, dtype="uint32")
    #     coord_dataset.attrs.update(attributes)

    #     # Add JPEG tables if present
    #     if jpeg_tables is not None:
    #         f.create_dataset(
    #             "jpeg_tables",
    #             data=np.frombuffer(jpeg_tables, dtype=np.uint8),
    #             dtype="uint8",
    #         )

    #     # Store metadata as attributes
    #     f.attrs["original_tile_size"] = original_tile_size
    #     f.attrs["target_tile_size"] = patch_size
    #     f.attrs["width"] = slide_width
    #     f.attrs["height"] = slide_height
    #     f.attrs["num_tiles"] = len(tiles)
    #     f.attrs["aggregated"] = original_tile_size != patch_size
    #     f.attrs["use_jpeg_tables"] = jpeg_tables is not None



debugging_input = get_debugging_input(
    slide_path = Path("/mnt/oskar/slides/h2023007068t1-a-2_131136.svs"),
    patches_path = Path("/mnt/oskar/test/10x_2048px_0px_overlap/patches/h2023007068t1-a-2_131136_patches.h5"),
    level=1,
)
save_entropy_patches(*debugging_input)

# show_hdf5(Path("/mnt/oskar/entropy/10x_2048px_0px_overlap/visualization/h2023007068t1-a-2_131136.jpg"))
# coords attributes: <HDF5 dataset "coords": shape (98, 2), type "<i8">
# level0_magnification: 40
# name: h2023007068t1-a-2_131136
# overlap: 0
# patch_size: 2048
# patch_size_level0: 8192
# savetodir: /mnt/oskar/test/10x_2048px_0px_overlap
# target_magnification: 10
